# FINA 4713 – Project Skeleton Code
**Introduction to AI and Big Data in Finance**

This notebook walks through a complete pipeline for cross-sectional return prediction:
preprocessing → model selection → expanding-window retraining → portfolio construction → evaluation.
Use it as a starting point. Replace or extend any step for your group's exploration direction.

| Split | Period |
|-------|--------|
| Training | Jan 2005 – Dec 2015 |
| Validation | Jan 2016 – Dec 2018 |
| Test | Jan 2019 – Dec 2024 |

## Setup

### Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy.stats import spearmanr
from sklearn.linear_model import Ridge
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import warnings; warnings.filterwarnings('ignore')

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(iterable, **kwargs): return iterable

## 1. Load Data

### Load and inspect

In [ ]:
df = pd.read_parquet('jkp_data_us_slim.parquet')
print(f"Shape : {df.shape}")
print(f"Period: {df['eom'].min().date()} to {df['eom'].max().date()}")
print(f"Firms : {df['id'].nunique():,}")
print(f"\nColumns:\n{df.columns.tolist()}")

## 2. Feature Selection and Train/Val/Test Split

### Define features, log-transform size, split by date

In [ ]:
TARGET = 'ret_exc_lead1m'

# Log-transform size: market equity is right-skewed
df['log_me'] = np.log1p(df['me'].clip(lower=0))

# Use all numeric characteristics in the slim file (plus log_me)
META     = ['id', 'eom', 'excntry', TARGET, 'me']
FEATURES = [c for c in df.columns if c not in META] + ['log_me']

# Drop rows with missing target (these cannot be used for training or evaluation)
df = df.dropna(subset=[TARGET])

# Temporal splits — do not change these dates; they are uniform across all groups
TRAIN_END = '2015-12-31'
VAL_END   = '2018-12-31'

train = df[df['eom'] <= TRAIN_END]
val   = df[(df['eom'] > TRAIN_END) & (df['eom'] <= VAL_END)]
test  = df[df['eom'] > VAL_END]

print(f"Train : {len(train):>7,} obs  ({train['eom'].min().date()} – {train['eom'].max().date()})")
print(f"Val   : {len(val):>7,} obs  ({val['eom'].min().date()} – {val['eom'].max().date()})")
print(f"Test  : {len(test):>7,} obs  ({test['eom'].min().date()} – {test['eom'].max().date()})")
print(f"Features: {len(FEATURES)}")

## 3. Preprocessing Pipeline

**No look-ahead:** winsorisation bounds, imputer medians, and scaler statistics
are computed on the **training set only** and then applied unchanged to validation and test.
In the expanding-window retraining step (Section 6), these objects are re-fit on each
expanded training window — still using only data available up to that point.

### Fit on training data; apply to val/test with the same parameters

In [ ]:
def fit_preprocessor(X_df):
    """Fit winsorisation bounds, median imputer, and z-score scaler on X_df."""
    low  = X_df.quantile(0.01)
    high = X_df.quantile(0.99)
    X_clipped = X_df.clip(lower=low, upper=high, axis=1)
    imp = SimpleImputer(strategy='median').fit(X_clipped)
    sc  = StandardScaler().fit(imp.transform(X_clipped))
    return low, high, imp, sc

def apply_preprocessor(X_df, low, high, imp, sc):
    """Apply pre-fitted preprocessor — no parameters are re-estimated here."""
    X = X_df.clip(lower=low, upper=high, axis=1)
    X = imp.transform(X)
    return sc.transform(X)

# Convert to numeric (coerce any non-numeric entries to NaN)
to_num = lambda df_: df_[FEATURES].apply(pd.to_numeric, errors='coerce')

# Fit once on training data
low, high, imp, sc = fit_preprocessor(to_num(train))

X_tr  = apply_preprocessor(to_num(train), low, high, imp, sc)
X_val = apply_preprocessor(to_num(val),   low, high, imp, sc)
X_te  = apply_preprocessor(to_num(test),  low, high, imp, sc)

y_tr  = train[TARGET].values
y_val = val[TARGET].values
y_te  = test[TARGET].values

print(f"X_train : {X_tr.shape}")
print(f"X_val   : {X_val.shape}")
print(f"X_test  : {X_te.shape}")

## 4. Baselines

### Historical-average benchmark — R² = 0 by construction

Predict every stock's return as the training-period mean.
We use the **Campbell & Thompson (2008) OOS R²** definition:

$$R^2_{OOS} = 1 - \frac{\text{MSE}_{\text{model}}}{\text{MSE}_{\text{hist avg}}}$$

Under this definition the historical-average model scores exactly **0.0** by construction.
Positive values mean the model beats the null; negative values mean it is worse.

In [ ]:
def oos_r2(y_true, y_pred, y_null):
    """OOS R² relative to the historical-average null (Campbell & Thompson 2008)."""
    return 1 - np.mean((y_true - y_pred)**2) / np.mean((y_true - y_null)**2)

hist_avg    = y_tr.mean()                       # scalar: training-period mean return
y_null_val  = np.full(len(y_val), hist_avg)
y_null_te   = np.full(len(y_te),  hist_avg)

print(f"Training mean return: {hist_avg:.5f}")
print(f"Historical-average  val R²: {oos_r2(y_val, y_null_val, y_null_val):+.4f}  (always 0.0)")
print(f"Historical-average test R²: {oos_r2(y_te,  y_null_te,  y_null_te):+.4f}  (always 0.0)")

### Market portfolio benchmark

Equal-weighted return of all stocks in the cross-section each month.
This is a long-only benchmark; it will be computed alongside the Ridge portfolio in Section 7.

## 4b. Portfolio Construction Helpers

These helpers are needed for Sharpe-based hyperparameter tuning in Section 5.
They are also used in the full portfolio evaluation in Section 7.

`portfolio_weights` converts raw predictions into dollar-neutral long–short weights using ranks.
`compute_portfolio_metrics` builds a monthly return series from those weights and returns
annualised return, vol, and Sharpe.

In [ ]:
def portfolio_weights(pred, max_w=0.05):
    """Rank-based dollar-neutral long–short weights, capped at max_w per stock."""
    n  = len(pred)
    w  = pd.Series(pred).rank() - (n + 1) / 2   # centred ranks
    w /= w.abs().sum()                            # normalise
    w  = w.clip(-max_w, max_w)                   # cap concentration
    w /= w.abs().sum()                            # re-normalise after cap
    return w.values

def compute_portfolio_metrics(df_eval, pred_col, target_col='ret_exc_lead1m', date_col='eom'):
    """Build monthly long–short returns and return (ann_ret, ann_vol, sharpe)."""
    monthly_returns = []
    for _, grp in df_eval.groupby(date_col):
        w = portfolio_weights(grp[pred_col].values)
        monthly_returns.append((w * grp[target_col].values).sum())
    pf = pd.Series(monthly_returns)
    ann_ret = pf.mean() * 12
    ann_vol = pf.std() * np.sqrt(12)
    sharpe  = ann_ret / ann_vol if ann_vol > 0 else 0.0
    return ann_ret, ann_vol, sharpe

In [ ]:
# Use this (or other variations) if you want to enforce zero net investment

# def portfolio_weights(pred, max_w=0.05):
#     s = pd.Series(pred).rank(method='average') - (len(pred) + 1) / 2
#     w = s.to_numpy(dtype=float)

#     # scale raw signal
#     long = np.clip(w, 0, None)
#     short = np.clip(-w, 0, None)
#     if long.sum() == 0 or short.sum() == 0:
#         return np.zeros_like(w)
#     w = 0.5 * long / long.sum() - 0.5 * short / short.sum()

#     # optional cap
#     w = np.clip(w, -max_w, max_w)

#     # re-enforce 50/50 long-short after cap
#     long = np.clip(w, 0, None)
#     short = np.clip(-w, 0, None)
#     if long.sum() == 0 or short.sum() == 0:
#         return np.zeros_like(w)
#     return 0.5 * long / long.sum() - 0.5 * short / short.sum()

## 5. Ridge Regression

### Hyperparameter tuning on the validation set

Search over a log-spaced grid of the regularisation parameter α.
The best α is the one maximising the **validation-set Sharpe ratio** of the long–short portfolio.
The test set is **not touched** at this stage.

In [ ]:
alpha_grid = [1e2, 1e3, 1e4, 1e5, 1e6, 1e7, 1e8]

print(f"{'Alpha':>10} | {'Val Sharpe':>10} | {'Val Ann. Ret':>12}")
print("-" * 38)

val_metrics = {}
val_eval = val[['eom', TARGET]].copy()

for alpha in alpha_grid:
    m = Ridge(alpha=alpha).fit(X_tr, y_tr)
    val_eval['ridge_pred'] = m.predict(X_val)
    ann_ret, ann_vol, sharpe = compute_portfolio_metrics(val_eval, 'ridge_pred', TARGET, 'eom')
    val_metrics[alpha] = sharpe
    print(f"{alpha:>10.0f} | {sharpe:>10.4f} | {ann_ret:>12.2%}")

best_alpha = max(val_metrics, key=val_metrics.get)
print(f"\n-> Best α = {best_alpha:.0f}  (val Sharpe = {val_metrics[best_alpha]:.4f})")

### Validation-set evaluation with best α

Confirm performance on the validation set before moving to the expanding-window step.

In [ ]:
ridge_val = Ridge(alpha=best_alpha).fit(X_tr, y_tr)
print(f"Ridge (α={best_alpha:.0f})")
print(f"  Val  R²: {oos_r2(y_val, ridge_val.predict(X_val), y_null_val):+.4f}")
print(f"  Test R²: {oos_r2(y_te,  ridge_val.predict(X_te),  y_null_te):+.4f}  ← peek for reference only")

## 6. Expanding-Window Retraining

A static model trained on 2005–2015 ignores six years of new data by the time it predicts 2024 returns.
The expanding window fixes this: every January we refit the **preprocessor and model coefficients**
on all data available up to that point.

The hyperparameter α is **not re-tuned** — it is fixed at the value selected in Section 5.
Re-tuning at each step would require a fresh inner validation window and adds substantial complexity;
for this project, fixed α is the standard and defensible choice.

### Retrain annually; collect test-period predictions

In [ ]:
records = []   # will collect one row per (stock, month) in the test period

for year in tqdm(range(2019, 2025), desc="Walk-Forward Years"):
    train_cut  = pd.Timestamp(f'{year}-01-01')
    test_start = pd.Timestamp(f'{year}-01-01')
    test_end   = pd.Timestamp(f'{year + 1}-01-01')

    df_tr = df[df['eom'] < train_cut]
    df_te = df[(df['eom'] >= test_start) & (df['eom'] < test_end)].dropna(subset=[TARGET])
    if df_te.empty:
        continue

    # Re-fit preprocessor on the expanded training window (no look-ahead)
    lo, hi, imp_r, sc_r = fit_preprocessor(to_num(df_tr))
    Xtr = apply_preprocessor(to_num(df_tr), lo, hi, imp_r, sc_r)
    Xte = apply_preprocessor(to_num(df_te), lo, hi, imp_r, sc_r)
    ytr = df_tr[TARGET].values

    # Refit model with fixed α
    ridge = Ridge(alpha=best_alpha).fit(Xtr, ytr)

    tmp = df_te[['eom', TARGET]].copy().reset_index(drop=True)
    tmp['ridge_pred'] = ridge.predict(Xte)
    tmp['hist_avg']   = ytr.mean()          # expanding training mean at this update
    records.append(tmp)

    print(f"  {year}: train {len(df_tr):>7,} obs  →  test {len(df_te):>6,} obs")

results = pd.concat(records, ignore_index=True)
print(f"\nTotal test observations: {len(results):,}")

## 7. Portfolio Construction

### Rank-based zero-cost long–short weights

Raw predicted returns can be extreme outliers. Replacing predictions with their cross-sectional rank
makes weights robust: the highest-ranked stock gets the same fixed overweight regardless of
whether its predicted return is 5% or 500%.  Steps:

1. Rank stocks within each month-cross-section.
2. Centre ranks → the portfolio is zero-cost (long leg ≈ short leg).
3. Normalise so |long| = |short| = 0.5.
4. Hard-cap at `max_w` per stock (rarely binds in large cross-sections).

In [ ]:
def portfolio_weights(pred, max_w=0.05):
    n   = len(pred)
    w   = pd.Series(pred).rank() - (n + 1) / 2   # centred ranks
    w  /= w.abs().sum()                            # normalise
    w   = w.clip(-max_w, max_w)                    # cap
    w  /= w.abs().sum()                            # re-normalise after cap
    return w.values

monthly = []
for eom_date, grp in results.groupby('eom'):
    ret_xs = grp[TARGET].values
    monthly.append({
        'date'   : eom_date,
        'Ridge'  : (portfolio_weights(grp['ridge_pred'].values) * ret_xs).sum(),
        'HistAvg': 0.0,              # no signal → zero long–short return (exact by construction)
        'Market' : ret_xs.mean(),    # equal-weighted long-only benchmark
    })

pf = pd.DataFrame(monthly).set_index('date').sort_index()
pf.index = pd.to_datetime(pf.index)
print(f"Monthly portfolio returns: {len(pf)} months")

## 8. Evaluation

### Out-of-sample R² and Information Coefficient (IC)

**OOS R²** (Campbell-Thompson): measures whether the model's predictions reduce MSE relative to
the historical average.  Values are typically small — even +0.5% is economically meaningful
for monthly returns.

**IC**: cross-sectional Spearman rank correlation between predicted and realised returns,
computed each month and averaged.  Measures ordinal predictability; positive IC means
high-predicted stocks tend to realise higher returns.

In [ ]:
y_true = results[TARGET].values
y_null = results['hist_avg'].values

# OOS R²
print("OOS R² (Campbell-Thompson, test 2019–2024)")
print(f"  Historical average : {oos_r2(y_true, y_null,                        y_null):+.4f}  (always 0.0)")
print(f"  Ridge              : {oos_r2(y_true, results['ridge_pred'].values,   y_null):+.4f}")

# Monthly IC
ic_ridge = results.groupby('eom').apply(
    lambda g: spearmanr(g['ridge_pred'], g[TARGET])[0]
)
t_stat = ic_ridge.mean() / ic_ridge.std() * np.sqrt(len(ic_ridge))

print(f"\nIC (cross-sectional Spearman ρ, monthly)")
print(f"  Ridge  mean IC : {ic_ridge.mean():+.4f}")
print(f"  Ridge  IC t-stat: {t_stat:+.2f}  ({len(ic_ridge)} months)")

### Sharpe ratio and portfolio summary

In [ ]:
ann_ret = pf.mean() * 12
ann_vol = pf.std()  * np.sqrt(12)
sharpe  = ann_ret / ann_vol

print(f"Portfolio Performance (annualised, test period 2019–2024)")
print(f"{'Model':<12} {'Ann. Return':>12} {'Ann. Vol':>10} {'Sharpe':>8}")
print("-" * 46)
for m in ['HistAvg', 'Ridge', 'Market']:
    print(f"{m:<12} {ann_ret[m]:>12.2%} {ann_vol[m]:>10.2%} {sharpe[m]:>8.2f}")
print()
print("Note: HistAvg long–short return is identically 0 (no signal → uniform weights = flat).")
print("      Market is long-only; Ridge is zero-cost — volatilities are not directly comparable.")

### Cumulative return plot

In [ ]:
cum = (1 + pf).cumprod() - 1

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(cum.index, cum['Market'] * 100,
        color='#888888', linestyle='--', lw=1.4,
        label=f"Market   (SR = {sharpe['Market']:.2f})")
ax.plot(cum.index, cum['Ridge'] * 100,
        color='#d6604d', lw=2.0,
        label=f"Ridge    (SR = {sharpe['Ridge']:.2f})")
ax.axhline(0, color='black', lw=0.8, linestyle='--', alpha=0.4)

ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.set_ylabel('Cumulative excess return (%)')
ax.set_title(
    'Expanding-window Ridge vs. Market — test period (Jan 2019 – Dec 2024)\n'
    'Ridge: rank-based zero-cost long–short  |  Market: equal-weighted long-only'
)
ax.legend(framealpha=0.9)
ax.grid(axis='y', lw=0.4, alpha=0.5)
fig.tight_layout()
plt.show()

## 9. Feature Importance (Economic Reasoning)

The largest Ridge coefficients from the **final** walk-forward step (model trained through 2023,
predicting 2024 returns) indicate which characteristics drive the model's predictions most.
Positive coefficients → the model expects higher returns from stocks with high values of that
characteristic; negative → lower returns.

**Caveat:** coefficients reflect only the last retraining step and are on standardised features,
so they measure relative importance within the model, not causal effects on stock returns.

In [ ]:
coefs = pd.Series(ridge.coef_, index=FEATURES)
top_pos = coefs.nlargest(10)
top_neg = coefs.nsmallest(10)
top_features = pd.concat([top_pos, top_neg]).sort_values()

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#d6604d' if c < 0 else '#4393c3' for c in top_features.values]
top_features.plot(kind='barh', color=colors, ax=ax)
ax.set_title('Top 10 Positive and Negative Predictors (Final Ridge Model, trained through 2023)')
ax.set_xlabel('Coefficient value (standardised features)')
ax.grid(axis='x', lw=0.5, alpha=0.5)
fig.tight_layout()
plt.show()